# 原 Stage1 0bp SOXL 选择损失集中度 v2

修复前瞻价格时间语义及负损失日少于 top10 的机械陷阱，保留完整 944 日物理账户分量。

In [1]:
import os,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_original_0bp_selection_tail_audit_v2_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][x]==0 for x in ('new_market_requests','model_fits','native_accounts','bootstraps'))
old=root/plan['predecessor']['card'];assert sha(old/'plan.json')==plan['predecessor']['plan_sha256'] and sha(old/'summary.json')==plan['predecessor']['summary_sha256'] and sha(old/'failure_attribution.json')==plan['predecessor']['failure_attribution_sha256']
a=root/src['stage1_card'];b=root/src['component_card'];sa=json.loads((a/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert sha(a/'summary.json')==src['stage1_summary_sha256'] and sha(b/'summary.json')==src['component_summary_sha256']
assert sa['nas_evidence']['sha256']==src['stage1_NAS_archive_sha256'] and sb['nas_evidence']['sha256']==src['component_NAS_archive_sha256']
assert sha(a/'qlib_recorder'/src['stage1_experiment_id']/src['stage1_recorder_id']/'artifacts/inputs/frame')==src['stage1_frame_sha256']
assert sha(b/'qlib_recorder'/src['component_experiment_id']/src['component_recorder_id']/'artifacts/analysis/daily')==src['component_daily_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['calendar_provider_uri']),region='us',kernels=1)
with R.uri_context(str(a/'qlib_recorder')):frame=R.get_recorder(recorder_id=src['stage1_recorder_id'],experiment_id=src['stage1_experiment_id']).load_object('inputs/frame')
with R.uri_context(str(b/'qlib_recorder')):physical=R.get_recorder(recorder_id=src['component_recorder_id'],experiment_id=src['component_experiment_id']).load_object('analysis/daily')
assert len(frame)==len(physical)==944
for name in ('signal_date','trade_date','exit_date'):assert pd.DatetimeIndex(pd.to_datetime(frame[name])).equals(pd.DatetimeIndex(pd.to_datetime(physical[name])))
calendar=pd.DatetimeIndex(D.calendar(start_time=str(frame.signal_date.iloc[0]),end_time=str(frame.signal_date.iloc[-1]),freq='day'));assert len(calendar)==944 and calendar.equals(pd.DatetimeIndex(pd.to_datetime(frame.signal_date)))
d=pd.DataFrame({'signal_date':pd.to_datetime(frame.signal_date),'trade_date':pd.to_datetime(frame.trade_date),'exit_date':pd.to_datetime(frame.exit_date),'SOXL_selection_log':physical.SOXL_selection_log.astype(float),'cash_opportunity_log':physical.cash_opportunity_log.astype(float),'post_open_soxl_weight':physical.post_open_soxl_weight.astype(float)})
for asset in ('soxl','soxx'):
 for prefix in ('raw_',''):
  first=frame[prefix+asset+'_open'].astype(float);last=frame[prefix+asset+'_exit_open'].astype(float);assert np.isfinite(first).all() and np.isfinite(last).all() and (first>0).all() and (last>0).all()
  d[prefix+asset+'_forward_price_return']=last/first-1
for prefix in ('raw_',''):d[prefix+'relative_forward_price_return']=d[prefix+'soxl_forward_price_return']-d[prefix+'soxx_forward_price_return']
d['raw_normalized_relative_gap']=d.raw_relative_forward_price_return-d.relative_forward_price_return
assert d.notna().all().all() and abs(d.SOXL_selection_log.sum()-0.22593145822639604)<1e-10
static=pd.concat({'label':d[['SOXL_selection_log','cash_opportunity_log','raw_relative_forward_price_return','relative_forward_price_return']],'meta':d[['post_open_soxl_weight']]},axis=1);static.index=pd.MultiIndex.from_arrays([d.signal_date,['SOXL_SOXX_CASH']*len(d)],names=['datetime','instrument']);dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'diagnostic':(str(d.signal_date.iloc[0].date()),str(d.signal_date.iloc[-1].date()))});assert len(dataset.prepare('diagnostic',col_set=['label','meta']))==944
def describe(g):
 neg=g.loc[g.SOXL_selection_log.lt(0)].sort_values('SOXL_selection_log');w=(-neg.SOXL_selection_log).to_numpy(float);tot=float(w.sum());eff=float(tot*tot/(w*w).sum()) if len(w) else None
 return {'sessions':len(g),'selection_log_sum':float(g.SOXL_selection_log.sum()),'negative_days':len(neg),'positive_days':int(g.SOXL_selection_log.gt(0).sum()),'zero_days':int(g.SOXL_selection_log.eq(0).sum()),'negative_sum':float(neg.SOXL_selection_log.sum()),'positive_sum':float(g.loc[g.SOXL_selection_log.gt(0),'SOXL_selection_log'].sum()),'worst2_share_of_negative_magnitude':float(w[:2].sum()/tot) if len(w) else None,'worst5_share_of_negative_magnitude':float(w[:5].sum()/tot) if len(w) else None,'effective_negative_loss_days':eff,'raw_normalized_relative_price_mismatch_days':int(g.raw_normalized_relative_gap.abs().gt(1e-8).sum()),'mean_post_open_soxl_weight':float(g.post_open_soxl_weight.mean())}
quarters={str(q):describe(g) for q,g in d.groupby(d.trade_date.dt.to_period('Q'),sort=True)};assert len(quarters)==16
for q,value in [('2024Q3',0.012208238445748431),('2026Q2',0.289452743369704),('2026Q3',-0.14390082802460138)]:assert abs(quarters[q]['selection_log_sum']-value)<1e-10
q3=d.loc[d.trade_date.dt.to_period('Q').eq(pd.Period('2026Q3'))];neg_q3=q3.loc[q3.SOXL_selection_log.lt(0)].sort_values('SOXL_selection_log');assert len(q3)==64 and len(neg_q3)==quarters['2026Q3']['negative_days']
worst=d.loc[(d.trade_date>=pd.Timestamp('2023-08-01'))&(d.exit_date<=pd.Timestamp('2024-01-31'))];assert len(worst)==126
report={'decision':'descriptive_original_Stage1_selection_loss_sparse_days_forward_price_context_only','full':describe(d),'quarters':quarters,'original_worst126':describe(worst),'Q3_2026_negative_days':neg_q3.to_dict(orient='records'),'Q3_2026_all_relative_price_mismatch_days':q3.loc[q3.raw_normalized_relative_gap.abs().gt(1e-8)].to_dict(orient='records'),'Q3_2026_forward_price_context_repaired':True,'v1_prior_return_not_used':True,'no_independent_quote_certification':True,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0}
tracking=card/'qlib_recorder';assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset);rec.save_objects(artifact_path='analysis',daily=d,report=report);rec.log_metrics(calendar_days=944,quarters=16,q3_negative_days=len(neg_q3),q3_worst2_share=quarters['2026Q3']['worst2_share_of_negative_magnitude'],fits=0,accounts=0)
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'result':'result.json','new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Forward price ratios are saved original Stage1 price-plane inputs, not independent vendor validation or fee-net contribution.','Selection-loss dates are hindsight data-quality candidates, not a trading switch.','All 2023-2026 history is reused development feedback.']}
(card/'result.json').write_text(json.dumps(report,indent=2,default=str)+'\n');(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n');e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='selection-tail-v2-restore-') as t:
 archive=Path(t)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/daily') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');print({'Q3':quarters['2026Q3'],'worst126':report['original_worst126']})


2026/10/23 07:08:10 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[13711:MainThread](2026-10-23 07:08:11,292) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[13711:MainThread](2026-10-23 07:08:11,294) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[13711:MainThread](2026-10-23 07:08:11,294) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


...

[13711:MainThread](2026-10-23 07:08:11,365) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_original_0bp_selection_tail_audit_v2_20261023.


[13711:MainThread](2026-10-23 07:08:11,367) INFO - qlib.workflow - [exp.py:258] - Experiment 788481216699025954 starts running ...


[13711:MainThread](2026-10-23 07:08:11,452) INFO - qlib.workflow - [recorder.py:345] - Recorder e03c00f4017746bf942c1306dddbdedb starts running under Experiment 788481216699025954 ...


[13711:MainThread](2026-10-23 07:08:12,442) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{'Q3': {'sessions': 64, 'selection_log_sum': -0.14390082802460138, 'negative_days': 8, 'positive_days': 9, 'zero_days': 47, 'negative_sum': -0.14892133342817773, 'positive_sum': 0.005020505403576371, 'worst2_share_of_negative_magnitude': 0.5795018066977847, 'worst5_share_of_negative_magnitude': 0.9016053573631932, 'effective_negative_loss_days': 4.731758855992485, 'raw_normalized_relative_price_mismatch_days': 51, 'mean_post_open_soxl_weight': 0.02951763407327821}, 'worst126': {'sessions': 126, 'selection_log_sum': -0.12138086818958936, 'negative_days': 36, 'positive_days': 31, 'zero_days': 59, 'negative_sum': -0.2320298134249813, 'positive_sum': 0.1106489452353919, 'worst2_share_of_negative_magnitude': 0.1694995109057206, 'worst5_share_of_negative_magnitude': 0.3972972350366485, 'effective_negative_loss_days': 17.866476464416547, 'raw_normalized_relative_price_mismatch_days': 115, 'mean_post_open_soxl_weight': 0.1136343416880879}}


## 中文结论

# 原 Stage1 0bp SOXL 选择损失：2026Q3 少数持有日集中

从冻结完整 Stage1 A_0 的物理账户 Qlib 分量读取全部 944 日、16 个季度；对同一成交至退出区间重新计算原始与标准化价格比，不再把决策前历史收益误作前瞻收益。v1 的错误时间语义及“负损失仅 8 日却取 top10”机械门均保留为失败证据。v2 原生 Recorder `e03c00f4017746bf942c1306dddbdedb` 与 NAS SHA-256 `9d0fab5f8717b65d75388a9775855c709a5e053762da69b3f8eadce31566810e` 已定向读回及远端限定恢复，0 模型拟合、0 新账户。

全路径 SOXL 相对选择对数贡献 +0.225931，负贡献 336 日，最差五日仅占全路径负值绝对额 8.0%。**2026Q3 不同**：64 日中负值 8 日、正值 9 日、零值 47 日，季度净贡献 -0.143901。最差两日（7 月 6 日、7 月 27 日）占该季负值绝对额 58.0%；最差五日占 90.2%，有效负损失日数约 4.73。这定位为**7 月的集中 SOXL 持有损失**，不是整个季度每日持续输给 SOXX。原账户最差 126 日则有 36 个负选择日，最差五日只占 39.7%；不同失败区间需要同时解释。

7 月 6 日与 27 日持有段的 SOXL−SOXX 原价前瞻简单收益分别约 −12.04%/−14.76%，成交后 SOXL 权重约 29.9%/29.1%，物理选择对数贡献分别 −0.03904/−0.04726。两日原价与标准化价计算的相对收益差，在全部八个负损失日的最大绝对值仅 1.94e-07，未见足以解释上述跌幅的价格调整面不连续。内部 Qlib/账户闭合与 NAS 身份已通过；这**不等于**独立供应商报价或公司行动的外部认证。

补充只读风险状态：7 月 6 日原始名义 beta 与自有上限同为 1.330，状态0；7 月 27 日均为 1.318，状态3 且当时回撤约 20.7%，未触及 22% emergency 门。源码中的状态3是停损后完成爬坡的全 beta 持有阶段，**不是开盘订单 HOLD，也不是 emergency 降 beta**；两日都执行开盘目标，原始目标均未被上限压低。这是事后路径解释，不证明状态机或仓位造成亏损，也不能按 7 月日期制定开关。

决定 `descriptive_original_Stage1_selection_loss_sparse_days_forward_price_context_only`。下一研究问题是决策前能否识别类似的单日半导体下行风险，同时保留其他区间现金保护与有利 SOXL 暴露；需要与原 39 项及已关闭尾部/广度路线去重，先核来源与匹配信息门，再以完整 0bp Tiered 自有账户检验。当前 0bp SOXX 硬门、2/5bp 和独立前向仍未通过。
